In [1]:
!nvidia-smi

Tue Jul 28 06:04:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
!pip install -q -U datasets transformers accelerate peft trl sqlglot

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 162.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.0/719.0 kB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 56.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires sqlglot<25.21,>=23.4, but you have sqlglot 30.14.0 which is incompatible.


In [3]:
from datasets import load_dataset

train_dataset = load_dataset(
    "birdsql/bird23-train-filtered",
    split="train"
)

print("Number of samples:", len(train_dataset))
print("Columns:", train_dataset.column_names)
print(train_dataset[0])

README.md:   0%|          | 0.00/4.42k [00:00<?, ?B/s]

train-00000-of-00001.jsonl:   0%|          | 0.00/2.65M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6601 [00:00<?, ? examples/s]

Number of samples: 6601
Columns: ['db_id', 'question', 'evidence', 'SQL']
{'db_id': 'movie_platform', 'question': 'Who is the director of the movie Sex, Drink and Bloodshed?', 'evidence': "Sex, Drink and Bloodshed refers to movie title = 'Sex, Drink and Bloodshed';", 'SQL': "SELECT director_name FROM movies WHERE movie_title = 'Sex, Drink and Bloodshed'"}


In [4]:
import pandas as pd

df = train_dataset.to_pandas()

print("Shape:", df.shape)
print("\nNull values:")
print(df.isna().sum())

print("\nEmpty strings:")
for column in df.columns:
    empty_count = df[column].fillna("").astype(str).str.strip().eq("").sum()
    print(f"{column}: {empty_count}")

print("\nUnique databases:", df["db_id"].nunique())

duplicate_count = df.duplicated(
    subset=["db_id", "question", "evidence"]
).sum()
print("Duplicate inputs:", duplicate_count)

print("\nSamples per database:")
print(df["db_id"].value_counts().head(10))

Shape: (6601, 4)

Null values:
db_id       0
question    0
evidence    0
SQL         0
dtype: int64

Empty strings:
db_id: 0
question: 0
evidence: 472
SQL: 0

Unique databases: 69
Duplicate inputs: 0

Samples per database:
db_id
works_cycles              383
public_review_platform    256
movie_3                   223
mondial_geo               211
soccer_2016               190
books                     184
simpson_episodes          164
student_loan              162
hockey                    156
olympics                  156
Name: count, dtype: int64


In [5]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   48G  189G  20% /


In [6]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [7]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/execution-aligned-text-to-sql"
)
DATA_DIR = PROJECT_DIR / "data"
ZIP_PATH = DATA_DIR / "bird_train.zip"

DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Dataset path:", ZIP_PATH)

Project directory: /content/drive/MyDrive/execution-aligned-text-to-sql
Dataset path: /content/drive/MyDrive/execution-aligned-text-to-sql/data/bird_train.zip


In [8]:
import subprocess

download_url = (
    "https://bird-bench.oss-cn-beijing.aliyuncs.com/train.zip"
)

if ZIP_PATH.exists():
    print("压缩包已经存在，跳过下载：", ZIP_PATH)
else:
    subprocess.run(
        [
            "wget",
            "-c",
            download_url,
            "-O",
            str(ZIP_PATH),
        ],
        check=True,
    )
    print("下载完成：", ZIP_PATH)

压缩包已经存在，跳过下载： /content/drive/MyDrive/execution-aligned-text-to-sql/data/bird_train.zip


In [9]:
print("File exists:", ZIP_PATH.exists())
print("File size:", ZIP_PATH.stat().st_size / 1024**3, "GB")

File exists: True
File size: 8.306972267106175 GB


In [10]:
import shutil
from pathlib import Path

LOCAL_ZIP = Path("/content/bird_train.zip")
LOCAL_DATA = Path("/content/bird_data")

if not LOCAL_ZIP.exists():
    print("正在从 Drive 复制压缩包……")
    shutil.copy2(ZIP_PATH, LOCAL_ZIP)
else:
    print("本地压缩包已经存在。")

if not LOCAL_DATA.exists():
    LOCAL_DATA.mkdir(parents=True)
    subprocess.run(
        ["unzip", "-q", str(LOCAL_ZIP), "-d", str(LOCAL_DATA)],
        check=True,
    )
    print("解压完成：", LOCAL_DATA)
else:
    print("数据库已经解压，跳过。")

正在从 Drive 复制压缩包……
解压完成： /content/bird_data


In [18]:
from pathlib import Path

LOCAL_DATA = Path("/content/bird_data")

nested_zips = sorted(LOCAL_DATA.rglob("*.zip"))

print("找到的内层压缩包：")

for path in nested_zips:
    print(path)

找到的内层压缩包：
/content/bird_data/__MACOSX/train/._train_databases.zip
/content/bird_data/train/train_databases.zip


In [19]:
import subprocess

INNER_ZIP = next(
    path for path in nested_zips
    if path.name == "train_databases.zip"
)

TRAIN_DIR = INNER_ZIP.parent
DATABASE_DIR = TRAIN_DIR / "train_databases"

existing_databases = list(DATABASE_DIR.rglob("*.sqlite"))

if existing_databases:
    print("数据库已经解压，跳过。")
    print("现有数据库数量：", len(existing_databases))
else:
    print("正在解压数据库，这一步可能需要几分钟……")

    subprocess.run(
        [
            "unzip",
            "-q",
            "-o",
            str(INNER_ZIP),
            "-d",
            str(TRAIN_DIR),
        ],
        check=True,
    )

    print("数据库解压完成。")

正在解压数据库，这一步可能需要几分钟……
数据库解压完成。


In [21]:
database_files = sorted(
    LOCAL_DATA.rglob("*.sqlite")
)

print("SQLite database count:", len(database_files))

for path in database_files[:10]:
    print(path)

SQLite database count: 138
/content/bird_data/train/__MACOSX/train_databases/address/._address.sqlite
/content/bird_data/train/__MACOSX/train_databases/airline/._airline.sqlite
/content/bird_data/train/__MACOSX/train_databases/app_store/._app_store.sqlite
/content/bird_data/train/__MACOSX/train_databases/authors/._authors.sqlite
/content/bird_data/train/__MACOSX/train_databases/beer_factory/._beer_factory.sqlite
/content/bird_data/train/__MACOSX/train_databases/bike_share_1/._bike_share_1.sqlite
/content/bird_data/train/__MACOSX/train_databases/book_publishing_company/._book_publishing_company.sqlite
/content/bird_data/train/__MACOSX/train_databases/books/._books.sqlite
/content/bird_data/train/__MACOSX/train_databases/car_retails/._car_retails.sqlite
/content/bird_data/train/__MACOSX/train_databases/cars/._cars.sqlite


In [22]:
movie_database = [
    path for path in database_files
    if path.name == "movie_platform.sqlite"
]

print("movie_platform database:", movie_database)

movie_platform database: [PosixPath('/content/bird_data/train/train_databases/movie_platform/movie_platform.sqlite')]


In [23]:
import sqlite3
import pandas as pd

sample = train_dataset[0]
db_id = sample["db_id"]

matching_databases = [
    path for path in database_files
    if path.stem == db_id
]

if len(matching_databases) != 1:
    raise RuntimeError(
        f"Expected 1 database for {db_id}, "
        f"but found {len(matching_databases)}"
    )

DB_PATH = matching_databases[0]

print("Database:", DB_PATH)
print("Question:", sample["question"])
print("Evidence:", sample["evidence"])
print("Gold SQL:", sample["SQL"])

Database: /content/bird_data/train/train_databases/movie_platform/movie_platform.sqlite
Question: Who is the director of the movie Sex, Drink and Bloodshed?
Evidence: Sex, Drink and Bloodshed refers to movie title = 'Sex, Drink and Bloodshed';
Gold SQL: SELECT director_name FROM movies WHERE movie_title = 'Sex, Drink and Bloodshed'


In [24]:
connection = sqlite3.connect(
    f"file:{DB_PATH.resolve()}?mode=ro",
    uri=True
)

check_result = connection.execute(
    "PRAGMA quick_check"
).fetchone()[0]

print("Database check:", check_result)

schema_df = pd.read_sql_query(
    """
    SELECT name, sql
    FROM sqlite_master
    WHERE type = 'table'
      AND name NOT LIKE 'sqlite_%'
    ORDER BY name
    """,
    connection
)

print("\nTables:", schema_df["name"].tolist())

for _, row in schema_df.iterrows():
    print("\n" + "=" * 80)
    print(row["sql"])

Database check: ok

Tables: ['lists', 'lists_users', 'movies', 'ratings', 'ratings_users']

CREATE TABLE "lists"
(
    user_id                     INTEGER
        references lists_users (user_id),
    list_id                     INTEGER not null
        primary key,
    list_title                  TEXT,
    list_movie_number           INTEGER,
    list_update_timestamp_utc   TEXT,
    list_creation_timestamp_utc TEXT,
    list_followers              INTEGER,
    list_url                    TEXT,
    list_comments               INTEGER,
    list_description            TEXT,
    list_cover_image_url        TEXT,
    list_first_image_url        TEXT,
    list_second_image_url       TEXT,
    list_third_image_url        TEXT
)

CREATE TABLE lists_users
(
    user_id                 INTEGER not null ,
    list_id                 INTEGER not null ,
    list_update_date_utc    TEXT,
    list_creation_date_utc  TEXT,
    user_trialist           INTEGER,
    user_subscriber         INTEGER,
   

In [25]:
try:
    result_df = pd.read_sql_query(
        sample["SQL"],
        connection
    )

    print("Execution succeeded")
    print("Returned rows:", len(result_df))
    display(result_df)

finally:
    connection.close()

Execution succeeded
Returned rows: 1


,director_name
0,"Zvonimir Jurić, Boris T. Matic, Antonio Nuić"
